# Part 4 — Run it and read the evidence

> **Extra, read only.** This lecture notebook is kept with its saved outputs. It ran with anemoi-training 0.17 and anemoi-inference 0.12 on the lecture dataset, not with the course kernel, so read it but do not rerun it in class.

**Question:** what does `anemoi-training train` do on one GPU, and what does it leave behind? Slides 25–35.  
Prerequisites: the runbook environment (Phase 0). Section 1 needs no dataset, graph or GPU; section 6 reads the runbook's MLflow store.  
Tags: `# RUN` runs live · `# REVIEW` is part of the 30-minute review path.


In [1]:
# RUN — paths, environment
import inspect                                              # reads the source code of a Python function or class
import logging
import os
import re                                                   # regular expressions: pattern matching in text
import textwrap
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")                           # hide library deprecation notices
import torch
import pytorch_lightning as pl
import anemoi.training

for name in ("pytorch_lightning", "lightning_fabric"):
    logging.getLogger(name).setLevel(logging.ERROR)         # keep the Lightning cells quiet

# where things are: <repo>/data/anemoi-course, or the folder named in ANEMOI_COURSE_BASE
BASE = Path(os.environ.get("ANEMOI_COURSE_BASE", "../../../data/anemoi-course")).resolve()
RUNS = BASE / "runs"                                        # where the training run left its log, checkpoints and MLflow store

print("anemoi-training", anemoi.training.__version__)
print("pytorch-lightning", pl.__version__)
print("torch", torch.__version__)


anemoi-training 0.17.0
pytorch-lightning 2.6.6
torch 2.10.0+cu128


## 1 · What PyTorch Lightning does, and what anemoi adds (slide 25)

You write four methods on a `LightningModule` (`__init__`, `training_step`, `validation_step`, `configure_optimizers`); the `Trainer` runs the loops, device moves, `backward`, optimiser and scheduler steps, multi-GPU, and calls callbacks at fixed **hooks**.

### 1a · Native Lightning: the smallest complete example


In [2]:
# RUN — a complete Lightning training, part 1: the four methods you write, and a callback that prints the hooks
class NextStateModel(pl.LightningModule):
    # the model: next state = f_theta(state), where f_theta is one linear layer
    def __init__(self, n=8):
        # Lightning calls this once, when the model object is created
        super().__init__()
        self.f = torch.nn.Linear(n, n)                                   # f_theta: n numbers in, n numbers out

    def training_step(self, batch, batch_idx):
        # Lightning calls this once per training batch; the loss it returns is what Lightning back-propagates
        x, y = batch                                                     # x = the state now, y = the true next state
        y_pred = self.f(x)                                               # the model's guess of the next state
        loss = torch.nn.functional.mse_loss(y_pred, y)                   # mean squared error between guess and truth
        return loss

    def validation_step(self, batch, batch_idx):
        # Lightning calls this once per validation batch, with gradients switched off
        x, y = batch
        y_pred = self.f(x)
        val_loss = torch.nn.functional.mse_loss(y_pred, y)
        self.log("val_loss", val_loss)                                   # a logged number; Lightning averages it per epoch

    def configure_optimizers(self):
        # Lightning calls this once before training, to get the optimiser and its learning-rate schedule
        opt = torch.optim.AdamW(self.parameters(), lr=1e-2)              # the optimiser, starting learning rate 0.01
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=40)    # cosine decay over the 40 steps
        return [opt], [{"scheduler": scheduler, "interval": "step"}]     # "step": update the learning rate after every batch


class HookPrinter(pl.Callback):
    # a callback: the Trainer calls each method below at a fixed point of its loop (a "hook")
    def on_train_start(self, trainer, module):
        print("hook  on_train_start")

    def on_train_batch_end(self, trainer, module, out, batch, i):
        if i % 10 == 0:                                                  # only every tenth batch, to keep the output short
            lr = trainer.optimizers[0].param_groups[0]["lr"]             # the learning rate right now
            print(f"hook  on_train_batch_end  step {trainer.global_step}  loss {out['loss']:.3f}  lr {lr:.2e}")

    def on_validation_end(self, trainer, module):
        val_loss = trainer.callback_metrics["val_loss"]                  # the number logged by validation_step
        print(f"hook  on_validation_end   epoch {trainer.current_epoch}  val_loss {val_loss:.3f}")

    def on_train_end(self, trainer, module):
        print("hook  on_train_end")


**Expected:** nothing printed — the cell only defines the two classes. `NextStateModel` has the four methods of slide 25; `HookPrinter` has four hooks that print when the `Trainer` reaches them.


In [3]:
# RUN — a complete Lightning training, part 2: the data, the Trainer, and the fit
torch.manual_seed(0)                                                     # same random numbers every time the cell runs
X = torch.randn(400, 8)                                                  # 400 states, 8 numbers each
scale = torch.diag(torch.linspace(0.5, 1.5, 8))                          # a fixed 8 x 8 diagonal matrix
Y = X @ scale + 0.05 * torch.randn(400, 8)                               # next state = scaled state + a little noise

train_set = torch.utils.data.TensorDataset(X[:320], Y[:320])             # the first 320 (state, next state) pairs: training
val_set = torch.utils.data.TensorDataset(X[320:], Y[320:])               # the last 80 pairs: validation
train = torch.utils.data.DataLoader(train_set, batch_size=16)            # 320 / 16 = 20 batches per epoch
val = torch.utils.data.DataLoader(val_set, batch_size=16)

trainer = pl.Trainer(
    accelerator="gpu",                                                   # run on the GPU of this node
    devices=1,                                                           # one GPU
    max_epochs=2,                                                        # two passes over the training data
    callbacks=[HookPrinter()],                                           # our hook printer
    num_sanity_val_steps=0,                                              # skip Lightning's validation check before training
    logger=False,                                                        # no tracker, ...
    enable_checkpointing=False,                                          # ... no checkpoint files, ...
    enable_progress_bar=False,                                           # ... no progress bar, ...
    enable_model_summary=False,                                          # ... no model summary: only our hooks print
)
model = NextStateModel()
trainer.fit(model, train, val)                                           # the whole training loop runs inside this call
print("steps run:", trainer.global_step, "= 2 epochs x", len(train), "batches")


hook  on_train_start


hook  on_train_batch_end  step 1  loss 1.640  lr 9.98e-03
hook  on_train_batch_end  step 11  loss 1.268  lr 8.25e-03
hook  on_validation_end   epoch 0  val_loss 0.844
hook  on_train_batch_end  step 21  loss 1.021  lr 4.61e-03
hook  on_train_batch_end  step 31  loss 0.965  lr 1.20e-03
hook  on_validation_end   epoch 1  val_loss 0.761
hook  on_train_end
steps run: 40 = 2 epochs x 20 batches


**Expected:** hooks in loop order — `on_train_start`, `on_train_batch_end` every 10 steps (loss falling, learning rate on the cosine), `on_validation_end` per epoch, `on_train_end`; 40 steps (2 epochs × 20 batches).
`NextStateModel` never called `backward()` or `optimizer.step()`: the `Trainer` did. `anemoi-training train` is the same loop with every argument read from the configuration.

### 1b · The same four methods in anemoi-training 0.17

The `LightningModule` is the method, `SingleTraining` (`training.method`), not the task `Forecaster` as slide 25's pseudo-code says — see `change.md`, slide 25, finding 1.


In [4]:
# RUN — which anemoi class is the LightningModule, and the rollout loop inside its training step
from anemoi.training.train.methods.single import SingleTraining
from anemoi.training.tasks import Forecaster
from anemoi.training.data.datamodule import AnemoiDatasetsDataModule

# 1) the role of each class: a LightningModule, a LightningDataModule, or neither
classes = [(SingleTraining, "training.method"), (Forecaster, "task"), (AnemoiDatasetsDataModule, "data + dataloader")]
for cls, group in classes:
    if issubclass(cls, pl.LightningModule):
        role = "LightningModule"
    elif issubclass(cls, pl.LightningDataModule):
        role = "LightningDataModule"
    else:
        role = "plain Python object"
    print(f"{cls.__name__:26s} {role:22s} <- config group {group}")

# 2) which class in the inheritance chain of SingleTraining defines training_step
for cls in SingleTraining.__mro__:                                       # the class, then its parent, then the parent's parent, ...
    if "training_step" in vars(cls):                                     # vars(cls) = what this class defines itself
        defined_in = cls.__name__
        break
print("training_step / validation_step / configure_optimizers defined in:", defined_in, "(the base of SingleTraining)")

# 3) training_step calls self._step; these are the lines of _step that do the work (slide 28)
source = inspect.getsource(SingleTraining._step)                         # the source code of the method, as one string
lines = textwrap.dedent(source).splitlines()                             # one entry per line, indentation removed
keys = ("x = self.task.get_inputs", "for i, task_kwargs", "y_pred = self(x)", "y = self.task.get_targets",
        "self.compute_loss_metrics", "x = self.task.advance_input", "loss = loss +", "loss *=")
print(f"\nSingleTraining._step  (train/methods/single.py, {len(lines)} lines; the {len(keys)} that matter)")
for n, line in enumerate(lines, 1):                                      # n = line number inside the method
    if line.strip().startswith(keys):                                    # keep the line if it starts with one of the keys
        print(f"{n:3d}  {line}")


SingleTraining             LightningModule        <- config group training.method
Forecaster                 plain Python object    <- config group task
AnemoiDatasetsDataModule   LightningDataModule    <- config group data + dataloader
training_step / validation_step / configure_optimizers defined in: BaseTrainingModule (the base of SingleTraining)

SingleTraining._step  (train/methods/single.py, 47 lines; the 8 that matter)
 11      x = self.task.get_inputs(batch, data_indices=self.data_indices)
 14      for i, task_kwargs in enumerate(task_steps):
 15          y_pred = self(x)
 17          y = self.task.get_targets(batch, **task_kwargs)
 20              self.compute_loss_metrics,
 32              x = self.task.advance_input(
 42          loss = loss + loss_next
 46      loss *= 1.0 / len(task_steps)


**Expected:** `SingleTraining` is the `LightningModule`, `AnemoiDatasetsDataModule` the `LightningDataModule`, `Forecaster` neither; the four methods live in `BaseTrainingModule`.
`_step` is slide 28's loop: `task.get_inputs`, then per rollout step `self(x)`, `task.get_targets`, `compute_loss_metrics`, `task.advance_input`; the loss is divided by the number of steps.

| Native Lightning (1a) | anemoi-training 0.17 | Configuration key |
|---|---|---|
| `NextStateModel(pl.LightningModule)` | `SingleTraining(BaseTrainingModule)` | `training: single` |
| `self.f = Linear(...)` | `self.model = AnemoiModelInterface(...)` | `model`, `graph` |
| `training_step`: `mse_loss(f(x), y)` | `_step`: rollout loop, weighted MSE with scalers | `task.rollout`, `training.training_loss`, `training.scalers` |
| `configure_optimizers`: AdamW + cosine | `instantiate(optimizer)`, `instantiate(lr_scheduler)` | `training.optimization` |
| `DataLoader(TensorDataset(...))` | `AnemoiDatasetsDataModule` (sample windows of slide 6) | `data`, `dataloader` |
| `pl.Trainer(max_epochs=2, ...)` | `pl.Trainer(**from the configuration)` — next cell | `training`, `system.hardware`, `diagnostics` |

### 1c · The `Trainer(...)` call, argument by argument


In [5]:
# RUN — the one pl.Trainer(...) call in anemoi-training: which configuration key sets each argument
from anemoi.training.train.train import AnemoiTrainer

source = inspect.getsource(AnemoiTrainer.train)                          # the method that builds the Trainer, as one string
start = source.index("pl.Trainer(")                                      # where the call begins
end = source.index("        )", start)                                   # the closing parenthesis of the call
block = source[start:end]                                                # the call only

rows = []                                                                # (Trainer argument, what sets it)
for line in block.splitlines()[1:]:                                      # one line per argument, e.g. max_epochs=self.config.training.max_epochs,
    match = re.match(r"\s*(\w+)=(.+?),?\s*(#.*)?$", line)                # split "name=value," into name and value
    if not match:
        continue                                                         # a comment line inside the call
    arg = match.group(1)
    value = match.group(2)
    config_keys = re.findall(r"self\.config\.([\w.]+)", value)           # e.g. ["training.max_epochs"]
    via_getattr = re.search(r'getattr\(self\.config\.([\w.]+),\s*"(\w+)"', value)    # e.g. getattr(self.config.training, "max_steps")
    if via_getattr:
        set_by = via_getattr.group(1) + "." + via_getattr.group(2)
    elif config_keys:
        set_by = ", ".join(config_keys)
    elif value.startswith("self."):
        set_by = "property AnemoiTrainer." + value[5:]                   # value[5:] drops the "self."
    else:
        set_by = "literal " + value                                      # a fixed value, e.g. False
    rows.append((arg, set_by))

print(f"{'Trainer argument':26s} set by")
for arg, set_by in rows:
    print(f"{arg:26s} {set_by}")

n_properties = 0                                                         # how many arguments the trainer builds itself
for arg, set_by in rows:
    if set_by.startswith("property"):
        n_properties = n_properties + 1
print(f"\n{len(rows)} arguments; {n_properties} are properties the trainer builds itself")


Trainer argument           set by
accelerator                property AnemoiTrainer.accelerator
callbacks                  property AnemoiTrainer.callbacks
deterministic              training.deterministic
detect_anomaly             diagnostics.debug.anomaly_detection
strategy                   property AnemoiTrainer.strategy
devices                    system.hardware.num_gpus_per_node
num_nodes                  system.hardware.num_nodes
precision                  training.precision
max_epochs                 training.max_epochs
max_steps                  training.max_steps
logger                     property AnemoiTrainer.logger
profiler                   property AnemoiTrainer.profiler
log_every_n_steps          diagnostics.log.interval
limit_train_batches        dataloader.limit_batches.training
limit_val_batches          dataloader.limit_batches.validation
num_sanity_val_steps       training.num_sanity_val_steps
accumulate_grad_batches    training.accum_grad_batches
gradient_clip_v

**Expected:** 23 arguments. `max_epochs`, `precision`, `gradient_clip_*` come from `training.*`; `devices`, `num_nodes` from `system.hardware.*`; `limit_*_batches` from `dataloader.limit_batches.*`; `log_every_n_steps` from `diagnostics.log.interval`.
Five are properties: `accelerator`, `callbacks`, `strategy` (`DDPGroupStrategy`), `logger` (MLflow), `profiler`. `use_distributed_sampler=False`: the dataset shards per GPU itself (slide 29).

> **Checkpoint:** which single string links the log, the checkpoint folder and the tracker? — The run id (slide 25), created in `AnemoiTrainer.run_id` and handed to `AnemoiCheckpoint` and the MLflow logger.


## 2–5 · Pre-launch checks, launch, the startup log, checkpoints (slides 26, 27, 34, 35) — *to be written: notebook-design.md, nb4 items 1–3 and 5.*


## 6 · The experiment record: MLflow (slides 31, 32)

The run wrote an **offline** store under `runs/logs/mlflow/` (`diagnostics.log.mlflow.offline: True`): per run, `params/` (every leaf of the resolved configuration), `metrics/` (one file per series), `tags/`, `artifacts/`.
No server, no UI: the two cells below read the files with `MlflowClient`, and `scripts/04_mlflow_static.py` renders the run page (`results/04-mlflow-run.png`) the same way.


In [6]:
# REVIEW — open the offline store, list the runs of the course experiment, pick the one that has metrics
from mlflow.tracking import MlflowClient

STORE = RUNS / "logs" / "mlflow"                                         # the offline store: plain files, no server
client = MlflowClient(tracking_uri=f"file://{STORE}")
exp = client.get_experiment_by_name("anemoi-course")                     # diagnostics.log.mlflow.experiment_name
runs = client.search_runs(experiment_ids=[exp.experiment_id], order_by=["attributes.start_time ASC"])

print(f"experiment {exp.name}   {'run id':34s} {'run_name':24s} {'status':9s} #metrics #params")
for r in runs:
    run_name = r.data.tags.get("mlflow.runName", "")                     # diagnostics.log.mlflow.run_name
    n_metrics = len(r.data.metrics)                                      # how many metric series the run logged
    n_params = len(r.data.params)                                        # how many configuration leaves it recorded
    print(f"{'':21s}{r.info.run_id:34s} {run_name:24s} {r.info.status:9s} {n_metrics:8d} {n_params:7d}")

run = runs[0]                                                            # keep the run with the most metric series
for r in runs:                                                           # (a run that died before logging has none)
    if len(r.data.metrics) > len(run.data.metrics):
        run = r
RUN_ID = run.info.run_id
print("\nusing run", RUN_ID)

plain_series = []                                                        # metric names without a "/", e.g. lr-AdamW
n_per_variable = 0                                                       # metric names with a "/": one per variable
for key in sorted(run.data.metrics):
    if "/" in key:
        n_per_variable = n_per_variable + 1
    else:
        plain_series.append(key)
print("metric series:", plain_series, "+ val_data_mse_metric/data/<var>/1 for", n_per_variable, "variables")

P = run.data.params                                                      # one entry per leaf of the resolved configuration
for key in ("config.model.decoders.0.mapper.num_channels", "config.training.max_epochs", "config.training.optimization.lr"):
    print(f"parameter  {key} = {P[key]}")


experiment anemoi-course   run id                             run_name                 status    #metrics #params
                     ab2ab757cc554916aeea8733c66784aa   course-o48-gt128-1gpu    FINISHED        28     858

using run ab2ab757cc554916aeea8733c66784aa
metric series: ['epoch', 'lr-AdamW', 'rollout', 'train_multi_dataset_loss_epoch', 'train_multi_dataset_loss_step', 'val_data_lossvariablemapper_loss', 'val_multi_dataset_loss_epoch', 'val_multi_dataset_loss_step'] + val_data_mse_metric/data/<var>/1 for 20 variables
parameter  config.model.decoders.0.mapper.num_channels = 128
parameter  config.training.max_epochs = 5
parameter  config.training.optimization.lr = 6.25e-05


**Expected:** one run named `course-o48-gt128-1gpu`, `ab2ab757…`, `FINISHED`, with 28 metric series and 858 parameters — one per resolved configuration leaf, so runs can be filtered by any key (`num_channels = 128`, `max_epochs = 5`).
The logged names carry the dataset container and the optimiser: `train_multi_dataset_loss_step`, `val_multi_dataset_loss_epoch`, `lr-AdamW` — not the slide's `train_loss` / `val_mse` / `lr`. The per-variable `val_data_mse_metric/data/<var>/1` are slide 31's "first forecast-related evidence".


In [7]:
# REVIEW — the three curves of slide 31, part 1: read the logged values of each series
import matplotlib
matplotlib.use("Agg")                                                    # draw into memory, no window
import matplotlib.pyplot as plt


def series(key):
    # the logged history of one metric: the list of steps and the list of values
    history = client.get_metric_history(RUN_ID, key)                     # one entry per logged point
    steps = []
    values = []
    for point in history:
        steps.append(point.step)
        values.append(point.value)
    return steps, values


CURVES = ("train_multi_dataset_loss_step", "val_multi_dataset_loss_epoch", "lr-AdamW")
curves = {}                                                              # key -> (steps, values), used by the plot below
for key in CURVES:
    steps, values = series(key)
    curves[key] = (steps, values)
    rounded = []                                                         # the values, shortened for printing
    for v in values:
        if key.startswith("lr"):
            rounded.append(f"{v:.2e}")                                   # learning rates are tiny: scientific notation
        else:
            rounded.append(round(v, 3))
    print(f"{key:30s} {len(values)} points:", rounded)


train_multi_dataset_loss_step  5 points: [0.318, 0.127, 0.076, 0.069, 0.054]
val_multi_dataset_loss_epoch   5 points: [0.301, 0.122, 0.079, 0.063, 0.055]
lr-AdamW                       5 points: ['6.19e-06', '1.24e-05', '1.87e-05', '2.49e-05', '3.12e-05']


**Expected:** five points per series: the two losses fall, the learning rate rises. Five because `diagnostics.log.interval: 100` on a 500-step run.


In [8]:
# REVIEW — the three curves of slide 31, part 2: draw them side by side and save the figure
X_LABELS = ("step", "epoch", "step")                                     # what the x axis of each panel counts
fig, ax = plt.subplots(1, 3, figsize=(15, 4))                            # one panel per curve
for panel, key, x_label in zip(ax, CURVES, X_LABELS):
    steps, values = curves[key]
    if x_label == "epoch":
        x = range(len(values))                                           # epoch number 0..4
    else:
        x = steps                                                        # the global step, 100..500
    panel.plot(x, values, "o-")
    panel.set_title(key)
    panel.set_xlabel(x_label)
    panel.grid(alpha=.3)

lr_steps, lr_values = curves["lr-AdamW"]
ax[2].axhline(6.25e-5, ls="--", c="grey")                                # the learning rate the configuration asks for
ax[2].text(lr_steps[0], 6.25e-5, " optimization.lr (1 GPU), reached at warm-up step 1000", va="bottom", fontsize=8)
fig.suptitle(f"anemoi-course · run {RUN_ID[:8]} · 5 epochs x 100 batches, 1 GPU · logged every diagnostics.log.interval = 100 steps")
fig.tight_layout()

if Path("../../../lecture/results").is_dir():
    OUT = Path("../../../lecture/results/04-curves.png")                            # the course's results folder
else:
    OUT = Path("04-curves.png")                                          # otherwise next to the notebook
fig.savefig(OUT, dpi=100)
print("saved", OUT.resolve())
plt.show()


saved /gpfs/home/bsc/bsc437208/gitlab/climate-data-ai-anemoi/lecture/results/04-curves.png


**Expected:** training loss 0.32 → 0.054 over steps 100–500 (fast drop, then a slow slope — slide 31's first shape); validation loss 0.30 → 0.055 over five epochs; learning rate a straight ramp 6.2e-6 → 3.1e-5.
The run ends inside the 1,000-step warm-up and never reaches `optimization.lr` (slide 28). Five points per curve: `diagnostics.log.interval: 100` on a 500-step run; use `interval: 10` for the course.

> **Checkpoint:** two runs differ only in `task.rollout.maximum`. Which MLflow view decides which is better at day 5? — None of the loss views: the `RolloutEval` metrics or a real forecast verification (Part 5).
